# MALTO "Who Moves?" Report

## Summary
We raise one alarm for every person, vehicle or animal that goes somewhere, and stay silent for pixel changes that are not an object changing place. The system combines YOLO11x detectors (COCO-pretrained, plus a fine-tuned person model), a custom Kalman + Hungarian tracker, and a displacement decision rule. A track triggers an alarm once it is 1 second old and has moved more than its own box height from its start, issuing one alarm per track using only past frames. On the public leaderboard, this approach improved the baseline's F-score from 0.603 to 0.845.

## What "moving" means
In the labels, `moving` never returns to 0 once it becomes 1, so we judge a track by its entire history rather than a short temporal window. According to the evaluation rules, an alarm counts if it occurs between 1 second before motion starts and 3 seconds after the target leaves. So we wait for clear displacement evidence. Two people exiting a car demonstrate our boundary decision for moving:

- **train_a1669c8b d87 (18–31 s):** Exits, closes the door, stays within 0.3 box heights of start. Labelled as not moving (driver).
- **train_95366bf2 t0 (21–43 s):** Walks around the car to the other door (1.6 box heights away). Labelled as moving.

Our threshold of 1.0 box height lies between the two (0.3 vs 1.6), establishing the first condition of our decision rule.

## What must never alarm
Avoiding false alarms is critical, as a false alarm on known non-targets carries a double penalty (weight 2). Our detector natively filters out most background noise, while static objects fail our displacement rule. Key cases evaluated include:

- **Static objects, mostly parked vehicles (71% of labelled rows):** Their box centers stay within about 1% of the box height, far below the one box height our rule requires. In train_b60956c2, 55 parked cars raise zero alarms. However, extreme vehicle density can cause tracker ID switches that mimic movement.
- **Shadows & weather:** In train_c74d48c1 (~8 s) only a car's shadow crosses the frame, raising no alarm. Camera shake and swaying branches in train_ec0660f1 (104 disturbances) and snow in train_b8c34e8b (538 disturbances) produce zero false alarms.
- **In-place motion:** A worker beside a car in train_a1669c8b d91 (0–31 s) moves limbs without going anywhere and raises no alarm.
- **People cut off by the border:** In train_298a12e2, people really walk along the top edge, but only their lower half is inside the frame. They are not labelled as targets, yet raised three false alarms (12–16 px boxes) in the first 9 s. We therefore ignore border boxes under 20 px.

## Validation
Test labels are hidden and public submissions are limited per day, so we built a local pipeline mirroring the official metric to evaluate our choices. Local scores run higher than public, because training people are larger (median 59 px vs 37 px in our test detections) and our settings were tuned on these videos, so we only compare local scores with each other.

81 of 90 training videos come from two parking lots, while test scenes are unseen, so a random split of videos puts the same backgrounds on both sides and rewards settings that fit specific scenes. We therefore split by scene (Lot A: 47, Lot B: 34, Other: 9), kept a setting only if it did not hurt performance in any scene, and tested the fine-tuned detector on Lot B, which it never saw.

Labels also contain noise: train_95366bf2 t14 is labelled moving but its box moves at most 2 px, and train_b8c34e8b t13 (126 s) is an "invisible person" moving exactly with a car.

## Key choices and why
- **Three detector passes per frame.** At 10 fps, each frame goes through three passes merged by same-class NMS:
    1. **COCO YOLO11x on full frame:** General object detection across all classes.
    2. **2×2 overlapping tiles:** Magnifies small objects ~1.4×, boosting small-person detection (Lot B F 0.848 → 0.890, Public 0.804 → 0.814).
    3. **Fine-tuned YOLO11x (person boxes only):** Solves most of the small-person losses. Used alone, it cut misses on 11 hard Lot B videos from 54 to 23 but raised false alarms from 3 to 20, so we keep only its person boxes (confidence ≥ 0.25) and combine them with COCO.
- **Our own tracker on every frame.** Baseline links boxes every 0.5 s, breaking on fast objects. Our Kalman filter prediction + Hungarian matching boxes to tracks at 10 fps, raising train recall from 0.66 to 0.89 (e.g., keeping apart and alarming three pedestrians in train_c74d48c1 at 1.4–2.3 s, which the baseline missed).
- **Weak boxes only extend tracks.** Confident detections (≥ 0.4) start tracks; weak ones (0.1–0.4) only extend them, so distant or occluded people are not cut into pieces when confidence temporarily drops. Lowering the start threshold to 0.25 cut Lot B precision (0.92 → 0.87), and a looser variant lost on public (0.845 → 0.832).
- **At least 1 s of track age** (second condition of our decision rule). Eliminates transient false boxes without recall cost (Lot B F 0.909 → 0.921). Longer minimums (3 s) dropped recall (0.92 → 0.89) as tracks often initiate late.
- **How we measure displacement.** We use relative box heights to normalize scale across distances, treating a 30 px distant person and a 150 px nearby person uniformly. At the frame border, a box stretches rather than slides due to the part of the object still outside the frame. Its border-side edge stays fixed while its free edge follows the object, so the center moves only about half as far. For border boxes, we therefore also measure the free edge (e.g., in train_44a67e0a, a car's center moves only 0.85 box heights, but its front edge passes 1.0 at 12.8 s and raises the alarm).

## Error analysis and fine-tuning
On the 13 training videos most similar to the test set, initial error analysis revealed that two-thirds of our loss came from undetected small pedestrians (20–50 px). Injecting ground-truth boxes for targets under 40 px boosted F from 0.835 to 0.970, showing that detector recall was our primary bottleneck. To solve this, we fine-tuned YOLO11x using frames from Lot A and Other scenes (sampled every 2 s) and evaluated on unseen Lot B. Fine-tuning came with a trade-off: missed targets dropped significantly (63 to 39), but false alarms slightly increased (19 to 28). However, this trade-off clearly paid off. Overall F on unseen Lot B rose from 0.890 to 0.921, and Public LB increased from 0.814 to 0.845 (e.g., catching two of three ~30 px pedestrians in train_8805fd7d that COCO missed).

## What worked and what did not (public F)
Each step was kept only if it gained in scene-based CV and public confirmed the gain (one target moves public by about 0.005).

**Kept (Public Progression):** Baseline (0.603) → YOLO11x 1280px (0.703) → Our tracker + displacement rule (0.804) → 2×2 tiles (0.814) → Fine-tuned person detector (0.845).

**Dropped (Gained in CV, failed on Public):**

| Idea | CV | Public |
|---|---|---|
| Learned motion classifier instead of the rule | 0.906 → 0.911 | 0.804 → 0.797 |
| Cutting tracks at ID switches | 0.906 → 0.935 | 0.804 → 0.791 |
| 3×3 tiles for small people | 0.835 → 0.859 | 0.814 → 0.806 |
| Start 1 box, keep lost tracks 4 s | 0.933 → 0.942 | 0.814 → 0.810 |
| Vehicle score thresholding | 0.921 → 0.932 | 0.845 → 0.845 |
| Motion check on weak tracks | 0.921 → 0.942 | 0.845 → 0.845 |

**Lesson learned:** The dropped ideas likely fit training-scene traits (crowded lots with 10.8 cars/frame vs 4.4 on test; and larger people). For instance, the classifier suffered a false-alarm penalty of 29 on known lots but 108 on unseen ones. The motion check only added repeat alarms on targets already hit, yielding no score gain. Once the core tracker and displacement rules were established, the remaining errors were mostly small people the detector misses, so only detector changes (tiling, fine-tuning) gained further on public.

## Models, reproducibility, limits
- **Models & Compliance:** YOLO11x (`yolo11x.pt`, AGPL-3.0) fine-tuned on train frames. Faster R-CNN, RT-DETR R50, and YOLO11m were tried and dropped. No VLM, CLIP, open-vocabulary, or video models were used, and test videos were never watched or labelled.
- **Reproducibility:** Self-contained pipeline. Fine-tuning uses a fixed seed, 23 epochs, `freeze=10`, AdamW `lr=0.001`, `imgsz=1280`, and deterministic mode, reproducing the 0.845 Public LB exactly. We also selected a COCO-only model (0.814) as a deterministic second final submission backup.
- **Limits:** Main bottlenecks are for small people (20–45 px) and false alarms from person boxes on empty areas. The fine-tune model was validated on one unseen lot only (Lot B).
